<a href="https://colab.research.google.com/github/JoviWZhu/20206RAG/blob/RAG-AttributeFilter/FineWeb_Edu_Attribute_Filter_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets sentence-transformers faiss-cpu rank_bm25 huggingface_hub pymongo google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.5 MB/s eta 0:00:00


In [9]:
from google import genai
from google.genai import types
from google.colab import userdata


# ==========================================
# 1. INITIALIZATION & CLIENT SETUP
# ==========================================

# 1. Fetch your Gemini API key natively from Colab Secrets
GM_TOKEN_DEV = userdata.get('GM_TOKEN_DEV')

# 2. Initialize the official Google GenAI Client
client = genai.Client(api_key=GM_TOKEN_DEV)

system_prompt = "You are a helpful assistant."
user_prompt = "Hello!"

# 3. Generate content using the active production model name
response = client.models.generate_content(
    model='gemini-3.5-flash-lite', # <-- FIXED: Updated to the active production model
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        max_output_tokens=200 # <-- FIXED: Removed deprecated temperature mapping
    ),
    contents=user_prompt
)

# 4. Print out the text result cleanly
print(response.text)


Hello! How can I help you today?


In [3]:
import json
import random
import numpy as np
import faiss
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from huggingface_hub import InferenceClient
from google.colab import userdata


print("⚡ Step 1: Initializing SentenceTransformer & CrossEncoder models...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
reranker_model = CrossEncoder("BAAI/bge-reranker-base")

⚡ Step 1: Initializing SentenceTransformer & CrossEncoder models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [6]:
import json
import re
from typing import Literal
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

class MetadataExtraction(BaseModel):
    domain: Literal[
        'STEM',
        'Humanities',
        'Social Sciences',
        'Vocational & Applied Arts',
        'Language & Literature'
    ]
    sub_topic: str = Field(
        description="A granular 1-to-3 word keyword describing the core academic subject matter"
    )
    structural_type: Literal[
        'Definition',
        'Tutorial / How-To',
        'Research Abstract',
        'Historical Narrative',
        'Q&A / Practice Problem'
    ]
    target_audience: Literal[
        'Primary School',
        'Middle School',
        'High School',
        'University',
        'Professional / Academic'
    ]


def extract_metadata_with_llm(raw_text_chunk):
    """Calls the LLM to extract metadata using native structured output parsing."""
    system_prompt = (
        "You are a strict data transformation engine. Your job is to read an educational text chunk "
        "and extract specific metadata attributes according to the provided schema."
    )
    user_prompt = f'Input Text: "{raw_text_chunk}"'

    try:
        response = client.models.generate_content(
            model='gemini-3.5-flash-lite',  # Replace with your active model ID (e.g. gemini-2.5-flash, gemini-1.5-flash)
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                max_output_tokens=200,
                response_mime_type="application/json",
                response_schema=MetadataExtraction,
            ),
            contents=user_prompt
        )

        # Method 1: Use SDK's automatically parsed object if available
        if getattr(response, "parsed", None):
            return response.parsed.model_dump()

        # Method 2: Fallback manual string extraction if response.parsed is absent
        raw_json_str = (response.text or "").strip()

        if not raw_json_str:
            raise ValueError("API returned an empty text payload.")

        # Clean markdown code block fences if present
        cleaned_str = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_json_str, flags=re.MULTILINE).strip()

        return json.loads(cleaned_str)

    except Exception as e:
        print(f"⚠️ Metadata extraction fallback triggered: {e}")
        return {
            "domain": "STEM",
            "sub_topic": "General Science",
            "structural_type": "Definition",
            "target_audience": "High School"
        }

In [28]:
# ==========================================
# 3. DATASET INGESTION & INDEX BUILDING
# ==========================================
print("\n⚡ Step 3: Loading sample documents from FineWeb-Edu dataset...")
random_skip_offset = random.randint(0, 10000)
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
raw_dataset_head = dataset.skip(random_skip_offset).take(10)  # Reduced to 100 for quick execution

documents_store = []
print("⚡ Extracting metadata using Gemini LLM and generating embeddings...")

for idx, item in enumerate(raw_dataset_head):
    text = item["text"]

    # 1. Extract dynamic metadata using the fixed Gemini function
    metadata = extract_metadata_with_llm(text[:1000])  # Send first 1000 chars for efficiency

    # 2. Store structured payload
    documents_store.append({
        "id": idx,
        "text": text,
        "metadata": metadata
    })
    print(f"  [Doc {idx}] Domain: '{metadata.get('domain')}' | Sub-Topic: '{metadata.get('sub_topic')}'")

raw_texts = [doc["text"] for doc in documents_store]

# --- BUILD DENSE INDEX (FAISS) ---
print("⚡ Building FAISS Vector Index...")
embeddings = embedding_model.encode(raw_texts, convert_to_numpy=True)
faiss.normalize_L2(embeddings)
dimension = embeddings.shape[1]
vector_index = faiss.IndexFlatIP(dimension)
vector_index.add(embeddings)

# --- BUILD SPARSE INDEX (BM25) ---
print("⚡ Building BM25 Keyword Index...")
tokenized_corpus = [text.lower().split(" ") for text in raw_texts]
bm25_index = BM25Okapi(tokenized_corpus)


⚡ Step 3: Loading sample documents from FineWeb-Edu dataset...


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

⚡ Extracting metadata using Gemini LLM and generating embeddings...
  [Doc 0] Domain: 'STEM' | Sub-Topic: 'Chondroitin sulfate'
  [Doc 1] Domain: 'STEM' | Sub-Topic: 'Cambrian paleontology'
  [Doc 2] Domain: 'STEM' | Sub-Topic: 'SQL Aggregate Functions'
  [Doc 3] Domain: 'STEM' | Sub-Topic: 'Geology and Remote Sensing'
  [Doc 4] Domain: 'Social Sciences' | Sub-Topic: 'Agricultural Development'
  [Doc 5] Domain: 'STEM' | Sub-Topic: 'Runaway Greenhouse Effect'
  [Doc 6] Domain: 'Humanities' | Sub-Topic: 'Black History Month'
  [Doc 7] Domain: 'Humanities' | Sub-Topic: 'Haga Palace history'
  [Doc 8] Domain: 'Social Sciences' | Sub-Topic: 'Korean History and Economics'
  [Doc 9] Domain: 'STEM' | Sub-Topic: 'Genetics'
⚡ Building FAISS Vector Index...
⚡ Building BM25 Keyword Index...


In [31]:
# ==========================================
# 4. ATTRIBUTE-PREFILTERED HYBRID SEARCH
# ==========================================

def run_attribute_hybrid_rag_search(user_query, attribute_filter=None, top_n_candidates=5, final_top_k=2):
    """
    1. Pre-filters documents matching the target metadata criteria.
    2. Runs hybrid search (Vector + BM25) over the filtered pool.
    3. Reranks using CrossEncoder.
    4. Passes context to LLM for final generation via Google AI SDK.
    """
    print(f"\n🔍 Processing Search Query: '{user_query}'")
    if attribute_filter:
        print(f"🏷️ Metadata Filter Enforced: {attribute_filter}")

    # Helper function to enforce strict pre-filtering
    def passes_filter(doc_meta):
        if not attribute_filter:
            return True
        for key, required_val in attribute_filter.items():
            if doc_meta.get(key) != required_val:
                return False
        return True

    # --- PATHWAY A: DENSE SEMANTIC RETRIEVAL + PRE-FILTERING ---
    query_embedding = embedding_model.encode([user_query], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    overfetch_k = min(len(documents_store), 50)
    _, dense_indices = vector_index.search(query_embedding, overfetch_k)

    dense_candidate_ids = []
    for idx in dense_indices[0]:
        doc_meta = documents_store[idx]["metadata"]
        if passes_filter(doc_meta):
            dense_candidate_ids.append(documents_store[idx]["id"])
            if len(dense_candidate_ids) >= top_n_candidates:
                break

    # --- PATHWAY B: SPARSE KEYWORD RETRIEVAL + PRE-FILTERING ---
    tokenized_query = user_query.lower().split(" ")
    bm25_scores = bm25_index.get_scores(tokenized_query)
    sorted_bm25_indices = np.argsort(bm25_scores)[::-1]

    sparse_candidate_ids = []
    for idx in sorted_bm25_indices:
        if bm25_scores[idx] == 0:
            break
        doc_meta = documents_store[idx]["metadata"]
        if passes_filter(doc_meta):
            sparse_candidate_ids.append(documents_store[idx]["id"])
            if len(sparse_candidate_ids) >= top_n_candidates:
                break

    # --- CANDIDATE POOL FUSION ---
    fused_candidate_ids = list(set(dense_candidate_ids + sparse_candidate_ids))
    filtered_candidate_docs = [documents_store[doc_id] for doc_id in fused_candidate_ids]

    print(f"✅ Filtered down to {len(filtered_candidate_docs)} candidates matching metadata rules.")

    if not filtered_candidate_docs:
        print("❌ No candidates met the required attribute filters.")
        return

    # --- RERANKING CANDIDATES ---
    print("⚡ Reranking filtered pool using Cross-Encoder...")
    rerank_pairs = [[user_query, doc["text"]] for doc in filtered_candidate_docs]
    rerank_scores = reranker_model.predict(rerank_pairs)

    ranked_indices = np.argsort(rerank_scores)[::-1]
    final_contexts = [filtered_candidate_docs[idx] for idx in ranked_indices[:final_top_k]]

    # --- LLM GENERATION (GOOGLE AI SDK) ---
    context_str = "\n---\n".join([
        f"[Metadata Domain: {doc['metadata']['domain']} | Sub-Topic: {doc['metadata']['sub_topic']}]\n{doc['text']}"
        for doc in final_contexts
    ])

    system_prompt = (
        "You are an academic study assistant. Answer the user question using ONLY the provided text context. "
        "If the information is missing from the provided context, state: 'I cannot find the answer in the provided documents.'"
    )
    user_prompt = f"Textbook Context:\n{context_str}\n\nUser Question: {user_query}\n\nStudy Guide Answer:"

    try:
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                temperature=0.2,
                max_output_tokens=400,
            ),
            contents=user_prompt
        )
        print("\n📚 [FINAL GENERATED ANSWER]:")
        print(response.text)

    except Exception as e:
        print(f"❌ LLM completion error: {e}")

In [32]:

# ==========================================
# 5. EXECUTION EXAMPLE
# ==========================================
# Example 1: Search restricted ONLY to documents tagged in the 'STEM' domain
run_attribute_hybrid_rag_search(
    user_query="What are basic properties of cells and organisms?",
    attribute_filter={"domain": "STEM"}
)




🔍 Processing Search Query: 'What are basic properties of cells and organisms?'
🏷️ Metadata Filter Enforced: {'domain': 'STEM'}
✅ Filtered down to 6 candidates matching metadata rules.
⚡ Reranking filtered pool using Cross-Encoder...

📚 [FINAL GENERATED ANSWER]:
I cannot find the answer in the provided documents.


In [33]:
# ==========================================
# 5. EXECUTION EXAMPLE
# ==========================================
# Example 1: Search restricted ONLY to documents tagged in the 'STEM' domain
run_attribute_hybrid_rag_search(
    user_query="What are SQL Aggregate Functions?",
    attribute_filter={"domain": "STEM"}
)


🔍 Processing Search Query: 'What are SQL Aggregate Functions?'
🏷️ Metadata Filter Enforced: {'domain': 'STEM'}
✅ Filtered down to 6 candidates matching metadata rules.
⚡ Reranking filtered pool using Cross-Encoder...

📚 [FINAL GENERATED ANSWER]:
An aggregate function is a tool used to summarize large or small volumes of data. They allow you to perform calculations such as counting rows, finding average values, or determining the minimum or maximum values within a column.


Here is how PostgreSQL helps you implement Graph RAG, how to query it, and why it simplifies your architecture.


### Seamless Hybrid Search + Graph RAG + Security (The "Triple Join")

The real advantage of Postgres comes when you combine **Graph Traversal**, **Vector Similarity**, and **Security/Role-Based Access Control (RBAC)** in a single atomic query execution.
---

### Key Architectural Takeaways

| Feature | Dual System (FAISS + Neo4j) | Unified PostgreSQL Stack |
| --- | --- | --- |
| **Data Sync** | High risk of graph nodes becoming out of sync with vectors. | **Zero sync overhead** (Entities, edges, vectors, and chunks sit in one DB). |
| **Pre-Filtering & Security** | Must implement RBAC separately in both Neo4j and FAISS. | **Single-stage RBAC** enforced across vectors, text, and graph relations simultaneously. |
| **Infrastructure Complexity** | High (Managing 2-3 specialized database engines). | **Low** (Single production-ready database engine). |
| **RAG Query Pipeline** | Vector Lookup → App Code → Graph Query → Filter | Single unified SQL/CTE Query execution. |

Here is how to set up PostgreSQL to ingest your FineWeb-Edu dataset with embeddings and metadata, followed by the complete Python code to execute a combined Vector + Keyword + Pre-Filtered Hybrid Search.

In [3]:
# 1. Quiet update and install PostgreSQL + pgvector extension
!apt-get update -y > /dev/null 2>&1
!apt-get install -y postgresql postgresql-contrib postgresql-16-pgvector > /dev/null 2>&1

# 2. Explicitly initialize/create the default cluster if it missing
!pg_createcluster 16 main --start || true

# 3. Start the PostgreSQL service using standard service tools
!service postgresql start

# 4. Create the superuser role and target database
!su - postgres -c "psql -c \"CREATE USER colab_user WITH SUPERUSER PASSWORD 'colab_password';\""
!su - postgres -c "psql -c \"CREATE DATABASE my_rag_db OWNER colab_user;\""

Error: cluster configuration already exists
[ OK ]
CREATE ROLE
CREATE DATABASE


In [4]:
import psycopg2

# Connect to the newly created local PostgreSQL cluster
conn = psycopg2.connect(
    dbname="my_rag_db",
    user="colab_user",
    password="colab_password",
    host="127.0.0.1",
    port="5432"
)
cursor = conn.cursor()

# Enable the vector extension
cursor.execute("CREATE EXTENSION IF NOT EXISTS vector;")
conn.commit()

print("✅ Server successfully started! Connected to PostgreSQL + pgvector.")

✅ Server successfully started! Connected to PostgreSQL + pgvector.


In [10]:
import random
import psycopg2
from psycopg2.extras import Json
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# ==========================================
# 1. CONNECT TO YOUR COLAB POSTGRESQL DB
# ==========================================
conn = psycopg2.connect(
    dbname="my_rag_db",
    user="colab_user",
    password="colab_password",
    host="127.0.0.1",
    port="5432"
)
cursor = conn.cursor()

# ==========================================
# 2. CREATE TABLE SCHEMA & INDEXES
# ==========================================
print("⚡ Initializing database schema...")
setup_schema_query = """
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS document_chunks (
    id SERIAL PRIMARY KEY,
    text TEXT NOT NULL,
    metadata JSONB NOT NULL,
    embedding vector(384),
    text_tokens tsvector GENERATED ALWAYS AS (
        to_tsvector('english', text)
    ) STORED
);

CREATE INDEX IF NOT EXISTS idx_chunks_embedding
ON document_chunks USING hnsw (embedding vector_cosine_ops);

CREATE INDEX IF NOT EXISTS idx_chunks_keywords
ON document_chunks USING gin (text_tokens);

CREATE INDEX IF NOT EXISTS idx_chunks_metadata
ON document_chunks USING gin (metadata);
"""
cursor.execute(setup_schema_query)
conn.commit()

# ==========================================
# 3. LOAD MODEL & INGEST DATA
# ==========================================
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

random_skip_offset = random.randint(0, 10000)
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)
raw_dataset_head = dataset.skip(random_skip_offset).take(10)

print("⚡ Extracting metadata, embedding, and inserting into PostgreSQL...")

insert_query = """
INSERT INTO document_chunks (text, metadata, embedding)
VALUES (%s, %s, %s::vector);
"""

for idx, item in enumerate(raw_dataset_head):
    text = item["text"]

    # Extract dynamic metadata via your Gemini LLM function
    metadata = extract_metadata_with_llm(text[:1000])

    # Generate 384-dim embedding vector
    embedding = embedding_model.encode(text).tolist()

    # Insert record directly into Postgres
    cursor.execute(insert_query, (text, Json(metadata), embedding))

    print(f"  [Inserted Doc {idx}] Domain: '{metadata.get('domain')}' | Sub-Topic: '{metadata.get('sub_topic')}'")

conn.commit()
print("✅ Ingestion complete. Data is ready for hybrid search!")

⚡ Initializing database schema...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

⚡ Extracting metadata, embedding, and inserting into PostgreSQL...
  [Inserted Doc 0] Domain: 'STEM' | Sub-Topic: 'Chondroitin sulfate'
  [Inserted Doc 1] Domain: 'STEM' | Sub-Topic: 'Cambrian paleontology'
  [Inserted Doc 2] Domain: 'STEM' | Sub-Topic: 'SQL Aggregate Functions'
  [Inserted Doc 3] Domain: 'STEM' | Sub-Topic: 'Geological Remote Sensing'
  [Inserted Doc 4] Domain: 'Social Sciences' | Sub-Topic: 'Agricultural Development'
  [Inserted Doc 5] Domain: 'STEM' | Sub-Topic: 'Runaway Greenhouse Effect'
  [Inserted Doc 6] Domain: 'Humanities' | Sub-Topic: 'Black History Month'
  [Inserted Doc 7] Domain: 'Humanities' | Sub-Topic: 'Haga Palace History'
  [Inserted Doc 8] Domain: 'Social Sciences' | Sub-Topic: 'Korean economic divergence'
  [Inserted Doc 9] Domain: 'STEM' | Sub-Topic: 'Mendelian Genetics'
✅ Ingestion complete. Data is ready for hybrid search!


In [13]:
def run_postgres_hybrid_search(user_query, attribute_filter=None, top_k=5):
    """
    Executes single-stage hybrid search (Vector + BM25-style Keyword)
    with JSONB pre-filtering directly inside PostgreSQL.
    """
    # Generate vector embedding for input query
    query_vector = embedding_model.encode(user_query).tolist()

    # Format metadata filter into a JSON string for SQL if provided
    filter_json = Json(attribute_filter) if attribute_filter else None

    hybrid_sql = """
    WITH user_input AS (
            SELECT
                %s::vector AS q_vec,
                websearch_to_tsquery('english', %s) AS q_kw,
                %s::jsonb AS q_filter
        ),
        dense_hits AS (
            SELECT id, ROW_NUMBER() OVER (ORDER BY embedding <=> ui.q_vec) AS rank
            FROM document_chunks, user_input ui
            WHERE (ui.q_filter IS NULL OR metadata @> ui.q_filter)
            ORDER BY embedding <=> ui.q_vec
            LIMIT 20
        ),
        sparse_hits AS (
            SELECT id, ROW_NUMBER() OVER (ORDER BY ts_rank_cd(text_tokens, ui.q_kw) DESC) AS rank
            FROM document_chunks, user_input ui
            WHERE text_tokens @@ ui.q_kw
              AND (ui.q_filter IS NULL OR metadata @> ui.q_filter)
            ORDER BY ts_rank_cd(text_tokens, ui.q_kw) DESC
            LIMIT 20
        ),
        scored_results AS (
            SELECT
                dc.id,
                dc.text,
                dc.metadata,
                COALESCE(1.0 / (60 + d.rank), 0.0) + COALESCE(1.0 / (60 + s.rank), 0.0) AS rrf_score
            FROM dense_hits d
            FULL OUTER JOIN sparse_hits s ON d.id = s.id
            JOIN document_chunks dc ON dc.id = COALESCE(d.id, s.id)
        )
        -- Deduplicate identical text passages based on top score
        SELECT DISTINCT ON (text) id, text, metadata, rrf_score
        FROM scored_results
        ORDER BY text, rrf_score DESC
        LIMIT %s;
    """

    cursor.execute(hybrid_sql, (query_vector, user_query, filter_json, top_k))
    results = cursor.fetchall()

    print(f"\n🔍 Search Query: '{user_query}'")
    if attribute_filter:
        print(f"🏷️ Metadata Filter Enforced: {attribute_filter}")

    print(f"\n📚 Found Top {len(results)} Hybrid Matches:")
    for row in results:
        doc_id, text, meta, score = row
        print(f"\n--- [Score: {score:.4f} | ID: {doc_id} | Domain: {meta.get('domain')}] ---")
        print(f"{text[:200]}...")

    return results

# Example Usage:
run_postgres_hybrid_search(
    user_query="granitic intrusions and coastal winds in Chile",
    attribute_filter={"domain": "STEM"},
    top_k=3
)


🔍 Search Query: 'granitic intrusions and coastal winds in Chile'
🏷️ Metadata Filter Enforced: {'domain': 'STEM'}

📚 Found Top 3 Hybrid Matches:

--- [Score: 0.0164 | ID: 14 | Domain: STEM] ---
16. Granitic Intrusion, Chanaral, Chile
The light-toned circular area at the right center of the photograph is a Paleozoic granodiorite exposed near the Pacific coast of Chile at Chanaral. The Humbold...

--- [Score: 0.0145 | ID: 13 | Domain: STEM] ---
A couple of months or so ago, I began talking about SQL functions and what they're used for. This article picks up where I left off. It defines the two types of functions, and begins to explain them i...

--- [Score: 0.0139 | ID: 12 | Domain: STEM] ---
Cambrian predator had killer eyes
View to a kill A fearsome predator that swam in the Cambrian oceans was in fact a metre-long arthropod with killer vision, say researchers.
Palaeontologist Dr John Pa...


[(14,
  '16. Granitic Intrusion, Chanaral, Chile\nThe light-toned circular area at the right center of the photograph is a Paleozoic granodiorite exposed near the Pacific coast of Chile at Chanaral. The Humboldt current flowing northward along the coast gives rise to cool onshore breezes. As these move overland, they warm up, and are capable of carrying more water vapor; thus they have a very drying effect and are responsible for the long thin Atacama Desert, which extends all the way along the coast of South America from Central Chile to Ecuador. The desert is one of the driest in the world, and the absence of soil or vegetation makes it ideal for investigation using satellite remote sensing techniques.\nThe pale tones of the intrusion contrast sharply with the darker tones of the metamorphic basement, enabling rapid mapping. A swarm of mafic dikes cuts the intrusion, and some of the larger dikes, with northeast trend, are just visible. The valley just south of the intrusion contains 